In [6]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2) # 다시 10으로 바뀌어야 함
        self.leaky = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()

        print("======== x ========\n")
        print(x)

        current = self.fc(x)
        print()
        print("======== fully connected ========\n")
        print(self.fc)
        spike, mem = self.leaky(current, mem)
        print ("========after leaky========")
        print(mem)

        return spike, mem

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long)
oddY = torch.zeros(32, dtype=torch.long)

x = torch.cat((evenX, oddX), dim=0).float().to(device)
# remainderX = x % 2
#x = torch.cat((x, remainderX), dim=1).float().to(device)
y = torch.cat ((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(1):
    optimizer.zero_grad()

    spike_count, mem = model(x)
    loss = criterion(mem, y)

    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        prediction = mem.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")





======== x ========

tensor([[ 4., 12., 16., 14.,  0., 10., 10., 14.,  2., 12.],
        [ 4., 18., 10., 12.,  4.,  4.,  0.,  6., 12., 10.],
        [ 0., 10.,  4., 16.,  0., 14.,  6., 12.,  4., 18.],
        [14.,  0., 16., 18.,  4.,  4.,  8., 10., 16.,  6.],
        [ 6.,  6., 10., 14.,  2., 10.,  4.,  6.,  0.,  4.],
        [ 0., 16.,  4., 16., 16., 18., 18.,  6.,  0.,  6.],
        [12., 12., 16.,  6.,  8., 14.,  0., 10., 10., 10.],
        [ 2., 12.,  2.,  4., 12., 10.,  0.,  8.,  8., 10.],
        [10., 14., 14., 16.,  4., 10.,  4.,  4., 10., 12.],
        [18., 12., 12., 12., 12.,  8., 18.,  0., 12., 10.],
        [ 8., 18.,  0., 12., 12.,  4.,  0., 18., 12., 18.],
        [12., 16., 14., 14.,  0., 12., 10., 16., 16., 14.],
        [14.,  2.,  2.,  2.,  4., 18., 18.,  6., 10.,  0.],
        [18.,  8.,  2., 18., 12.,  0.,  6., 14.,  0.,  0.],
        [ 4., 10., 18., 10., 16.,  2.,  6.,  2.,  6.,  8.],
        [12.,  8., 12., 16.,  6., 18., 18.,  6., 16.,  2.],
        [16., 10., 

In [9]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 32) # 은닉층
        self.leaky = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(32, 2)
        self.leaky2 = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()
        mem2 = self.leaky2.init_leaky()

        current = self.fc(x)
        spike_count = x.new_zeros(x.size(0), 2)

        for i in range(10):
            spike, mem = self.leaky(current, mem)

            current2 = self.fc2(spike)
            spike2, mem2 = self.leaky2(current2, mem2)
            spike_count = spike_count + spike2
            # 주의: 위 코드는 새로운 tensor를 만들지만, 아래는 기존 텐서를 수정하기에 역전파에 문제가 발생할 수 있음
            # spike_count += spike

        return spike_count, mem2

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long) # 짝수
oddY = torch.zeros(32, dtype=torch.long) # 홀수

x = torch.cat((evenX, oddX), dim=0).float().to(device)
y = torch.cat((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(10):
    optimizer.zero_grad()

    spike_count, mem = model(x)
    loss = criterion(spike_count, y)

    loss.backward()
    optimizer.step()

    if epoch % 1 == 0:
        prediction = spike_count.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")





epoch: 1, loss: 0.6931, accuracy: 50.00%
epoch: 2, loss: 0.6931, accuracy: 50.00%
epoch: 3, loss: 0.6931, accuracy: 50.00%
epoch: 4, loss: 0.6931, accuracy: 50.00%
epoch: 5, loss: 0.6931, accuracy: 50.00%
epoch: 6, loss: 0.6931, accuracy: 50.00%
epoch: 7, loss: 0.6931, accuracy: 50.00%
epoch: 8, loss: 0.6931, accuracy: 50.00%
epoch: 9, loss: 0.6931, accuracy: 50.00%
epoch: 10, loss: 0.6931, accuracy: 50.00%


ㅠㅠ...

In [10]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 32) # 은닉층
        self.leaky = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(32, 2)
        self.leaky2 = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()
        mem2 = self.leaky2.init_leaky()

        current = self.fc(x)
        spike_count = x.new_zeros(x.size(0), 2)

        for i in range(10):
            spike, mem = self.leaky(current, mem)

            current2 = self.fc2(spike)
            spike2, mem2 = self.leaky2(current2, mem2)
            spike_count = spike_count + spike2
            # 주의: 위 코드는 새로운 tensor를 만들지만, 아래는 기존 텐서를 수정하기에 역전파에 문제가 발생할 수 있음
            # spike_count += spike

        return spike_count, mem2

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long) # 짝수
oddY = torch.zeros(32, dtype=torch.long) # 홀수

x = torch.cat((evenX, oddX), dim=0).float().to(device)
y = torch.cat((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(10):
    optimizer.zero_grad()

    spike_count, mem = model(x)
    loss = criterion(spike_count, y)

    loss.backward()
    optimizer.step()

    if epoch % 1 == 0:
        prediction = spike_count.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")



model.eval()

with torch.no_grad():
    spike_count, mem = model(x)

    print("Spike Count:")
    print(spike_count[:10])

    print("평균 Spike Count:")
    print(spike_count.mean(dim=0))

    print("Spike Count 동률 개수:")
    print((spike_count[:, 0] == spike_count[:, 1]).sum().item())

    print("막전위:")
    print(mem[:10])

epoch: 1, loss: 1.8930, accuracy: 48.44%
epoch: 2, loss: 1.7344, accuracy: 48.44%
epoch: 3, loss: 1.6283, accuracy: 48.44%
epoch: 4, loss: 1.5541, accuracy: 48.44%
epoch: 5, loss: 1.4732, accuracy: 48.44%
epoch: 6, loss: 1.4070, accuracy: 48.44%
epoch: 7, loss: 1.3212, accuracy: 50.00%
epoch: 8, loss: 1.2436, accuracy: 50.00%
epoch: 9, loss: 1.1893, accuracy: 50.00%
epoch: 10, loss: 1.1325, accuracy: 50.00%
Spike Count:
tensor([[2., 0.],
        [4., 0.],
        [0., 0.],
        [4., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [2., 0.],
        [0., 1.],
        [1., 0.]], device='mps:0')
평균 Spike Count:
tensor([1.2344, 0.0938], device='mps:0')
Spike Count 동률 개수:
26
막전위:
tensor([[ 0.7118, -0.6676],
        [ 1.2390, -1.2334],
        [-0.0510,  0.6506],
        [ 1.3915, -2.0676],
        [ 0.7340, -1.8764],
        [ 0.6313, -0.9135],
        [-1.1330,  0.0866],
        [ 0.3067, -0.1746],
        [-0.6807,  0.4045],
        [ 0.9127, -1.4830]], device='mps:0'

In [11]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 32) # 은닉층
        self.leaky = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(32, 2)
        self.leaky2 = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()
        mem2 = self.leaky2.init_leaky()

        current = self.fc(x)
        spike_count = x.new_zeros(x.size(0), 2)

        for i in range(10):
            spike, mem = self.leaky(current, mem)

            current2 = self.fc2(spike)
            spike2, mem2 = self.leaky2(current2, mem2)
            spike_count = spike_count + spike2
            # 주의: 위 코드는 새로운 tensor를 만들지만, 아래는 기존 텐서를 수정하기에 역전파에 문제가 발생할 수 있음
            # spike_count += spike

        return spike_count, mem2

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long) # 짝수
oddY = torch.zeros(32, dtype=torch.long) # 홀수

x = torch.cat((evenX, oddX), dim=0).float().to(device)
y = torch.cat((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(200): # epoch 늘리기
    optimizer.zero_grad()

    spike_count, mem = model(x)
    loss = criterion(spike_count, y)

    loss.backward()
    optimizer.step()

    if epoch % 1 == 0:
        prediction = spike_count.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")



model.eval()

with torch.no_grad():
    spike_count, mem = model(x)

    print("Spike Count:")
    print(spike_count[:10])

    print("평균 Spike Count:")
    print(spike_count.mean(dim=0))

    print("Spike Count 동률 개수:")
    print((spike_count[:, 0] == spike_count[:, 1]).sum().item())

    print("막전위:")
    print(mem[:10])

epoch: 1, loss: 0.7028, accuracy: 48.44%
epoch: 2, loss: 0.6931, accuracy: 50.00%
epoch: 3, loss: 0.6931, accuracy: 50.00%
epoch: 4, loss: 0.6931, accuracy: 50.00%
epoch: 5, loss: 0.6931, accuracy: 50.00%
epoch: 6, loss: 0.6931, accuracy: 50.00%
epoch: 7, loss: 0.6931, accuracy: 50.00%
epoch: 8, loss: 0.6931, accuracy: 50.00%
epoch: 9, loss: 0.6931, accuracy: 50.00%
epoch: 10, loss: 0.6872, accuracy: 51.56%
epoch: 11, loss: 0.6872, accuracy: 51.56%
epoch: 12, loss: 0.6872, accuracy: 51.56%
epoch: 13, loss: 0.6872, accuracy: 51.56%
epoch: 14, loss: 0.6872, accuracy: 51.56%
epoch: 15, loss: 0.6813, accuracy: 53.12%
epoch: 16, loss: 0.6724, accuracy: 54.69%
epoch: 17, loss: 0.6665, accuracy: 56.25%
epoch: 18, loss: 0.6665, accuracy: 56.25%
epoch: 19, loss: 0.6665, accuracy: 56.25%
epoch: 20, loss: 0.6636, accuracy: 56.25%
epoch: 21, loss: 0.6636, accuracy: 56.25%
epoch: 22, loss: 0.6636, accuracy: 56.25%
epoch: 23, loss: 0.6607, accuracy: 56.25%
epoch: 24, loss: 0.6607, accuracy: 56.25%
e

bb !!

In [12]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 32) # 은닉층
        self.leaky = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(32, 2)
        self.leaky2 = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()
        mem2 = self.leaky2.init_leaky()

        current = self.fc(x)
        spike_count = x.new_zeros(x.size(0), 2)

        for i in range(10):
            spike, mem = self.leaky(current, mem)

            current2 = self.fc2(spike)
            spike2, mem2 = self.leaky2(current2, mem2)
            spike_count = spike_count + spike2
            # 주의: 위 코드는 새로운 tensor를 만들지만, 아래는 기존 텐서를 수정하기에 역전파에 문제가 발생할 수 있음
            # spike_count += spike

        return spike_count, mem2

model = SNN().to(device)

evenX = torch.randint(0, 10, (32, 10)) * 2
oddX = torch.randint(0, 10, (32, 10)) * 2 + 1

evenY = torch.ones(32, dtype=torch.long) # 짝수
oddY = torch.zeros(32, dtype=torch.long) # 홀수

x = torch.cat((evenX, oddX), dim=0).float().to(device)
y = torch.cat((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(200): # epoch 늘리기
    optimizer.zero_grad()

    spike_count, mem = model(x)
    loss = criterion(spike_count, y)

    loss.backward()
    optimizer.step()

    if epoch % 1 == 0:
        prediction = spike_count.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")


### Generated code for testing

model.eval()

with torch.no_grad():
    # 새로운 짝수 데이터 1000개
    even_test = torch.randint(0, 10, (1000, 10)) * 2

    # 새로운 홀수 데이터 1000개
    odd_test = torch.randint(0, 10, (1000, 10)) * 2 + 1

    test_x = torch.cat(
        (even_test, odd_test), dim=0
    ).float().to(device)

    test_y = torch.cat((
        torch.ones(1000, dtype=torch.long),
        torch.zeros(1000, dtype=torch.long)
    )).to(device)

    spike_count, _ = model(test_x)
    prediction = spike_count.argmax(dim=1)

    accuracy = (prediction == test_y).float().mean()

    even_acc = (prediction[:1000] == 1).float().mean()
    odd_acc = (prediction[1000:] == 0).float().mean()

    print(f"Test Accuracy: {accuracy.item():.2%}")
    print(f"짝수 정확도: {even_acc.item():.2%}")
    print(f"홀수 정확도: {odd_acc.item():.2%}")

epoch: 1, loss: 1.1275, accuracy: 50.00%
epoch: 2, loss: 1.0500, accuracy: 50.00%
epoch: 3, loss: 1.0024, accuracy: 50.00%
epoch: 4, loss: 0.9728, accuracy: 50.00%
epoch: 5, loss: 0.9463, accuracy: 50.00%
epoch: 6, loss: 0.8884, accuracy: 50.00%
epoch: 7, loss: 0.8779, accuracy: 50.00%
epoch: 8, loss: 0.8297, accuracy: 53.12%
epoch: 9, loss: 0.7756, accuracy: 53.12%
epoch: 10, loss: 0.7404, accuracy: 53.12%
epoch: 11, loss: 0.7518, accuracy: 53.12%
epoch: 12, loss: 0.7362, accuracy: 54.69%
epoch: 13, loss: 0.7134, accuracy: 54.69%
epoch: 14, loss: 0.6941, accuracy: 54.69%
epoch: 15, loss: 0.6941, accuracy: 54.69%
epoch: 16, loss: 0.7058, accuracy: 54.69%
epoch: 17, loss: 0.6974, accuracy: 54.69%
epoch: 18, loss: 0.6737, accuracy: 54.69%
epoch: 19, loss: 0.6737, accuracy: 54.69%
epoch: 20, loss: 0.6737, accuracy: 54.69%
epoch: 21, loss: 0.6797, accuracy: 54.69%
epoch: 22, loss: 0.6766, accuracy: 54.69%
epoch: 23, loss: 0.6797, accuracy: 54.69%
epoch: 24, loss: 0.6797, accuracy: 54.69%
e

ㅠㅠ...

In [14]:
import torch
import torch.nn as nn
import snntorch as snn

device = torch.device("mps")

class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 32) # 은닉층
        self.leaky = snn.Leaky(beta=0.9)
        self.fc2 = nn.Linear(32, 2)
        self.leaky2 = snn.Leaky(beta=0.9)

    def forward(self, x):
        mem = self.leaky.init_leaky()
        mem2 = self.leaky2.init_leaky()

        current = self.fc(x)
        spike_count = x.new_zeros(x.size(0), 2)

        for i in range(10):
            spike, mem = self.leaky(current, mem)

            current2 = self.fc2(spike)
            spike2, mem2 = self.leaky2(current2, mem2)
            spike_count = spike_count + spike2
            # 주의: 위 코드는 새로운 tensor를 만들지만, 아래는 기존 텐서를 수정하기에 역전파에 문제가 발생할 수 있음
            # spike_count += spike

        return spike_count, mem2

model = SNN().to(device)

evenY = torch.ones(320, dtype=torch.long) # 짝수
oddY = torch.zeros(320, dtype=torch.long) # 홀수

y = torch.cat((evenY, oddY), dim=0).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

model.train()

for epoch in range(200):
    optimizer.zero_grad()

    # 데이터셋 늘려주기
    evenX = torch.randint(0, 10, (320, 10)) * 2
    oddX = torch.randint(0, 10, (320, 10)) * 2 + 1
    x = torch.cat((evenX, oddX), dim=0).float().to(device)

    spike_count, mem = model(x)
    loss = criterion(spike_count, y)

    loss.backward()
    optimizer.step()

    if epoch % 1 == 0:
        prediction = spike_count.argmax(dim=1)
        accuracy = (prediction == y).float().mean()
        print(f"epoch: {epoch + 1}, loss: {loss.item():.4f}, accuracy: {accuracy.item():.2%}")


### Generated code for testing

model.eval()

with torch.no_grad():
    # 새로운 짝수 데이터 1000개
    even_test = torch.randint(0, 10, (1000, 10)) * 2

    # 새로운 홀수 데이터 1000개
    odd_test = torch.randint(0, 10, (1000, 10)) * 2 + 1

    test_x = torch.cat(
        (even_test, odd_test), dim=0
    ).float().to(device)

    test_y = torch.cat((
        torch.ones(1000, dtype=torch.long),
        torch.zeros(1000, dtype=torch.선long)
    )).to(device)

    spike_count, _ = model(test_x)
    prediction = spike_count.argmax(dim=1)

    accuracy = (prediction == test_y).float().mean()

    even_acc = (prediction[:1000] == 1).float().mean()
    odd_acc = (prediction[1000:] == 0).float().mean()

    print(f"Test Accuracy: {accuracy.item():.2%}")
    print(f"짝수 정확도: {even_acc.item():.2%}")
    print(f"홀수 정확도: {odd_acc.item():.2%}")

epoch: 1, loss: 1.4575, accuracy: 47.34%
epoch: 2, loss: 1.3576, accuracy: 47.50%
epoch: 3, loss: 1.2688, accuracy: 48.75%
epoch: 4, loss: 1.1810, accuracy: 48.44%
epoch: 5, loss: 1.0623, accuracy: 48.28%
epoch: 6, loss: 1.0454, accuracy: 49.38%
epoch: 7, loss: 0.9101, accuracy: 50.00%
epoch: 8, loss: 0.9036, accuracy: 49.69%
epoch: 9, loss: 0.9305, accuracy: 48.75%
epoch: 10, loss: 0.9024, accuracy: 47.03%
epoch: 11, loss: 0.8555, accuracy: 51.41%
epoch: 12, loss: 0.7930, accuracy: 50.31%
epoch: 13, loss: 0.7820, accuracy: 52.81%
epoch: 14, loss: 0.8071, accuracy: 49.38%
epoch: 15, loss: 0.7938, accuracy: 49.22%
epoch: 16, loss: 0.7481, accuracy: 51.56%
epoch: 17, loss: 0.7754, accuracy: 50.63%
epoch: 18, loss: 0.7467, accuracy: 50.63%
epoch: 19, loss: 0.7832, accuracy: 50.00%
epoch: 20, loss: 0.7605, accuracy: 50.63%
epoch: 21, loss: 0.7561, accuracy: 49.84%
epoch: 22, loss: 0.7482, accuracy: 51.72%
epoch: 23, loss: 0.7377, accuracy: 51.56%
epoch: 24, loss: 0.7219, accuracy: 50.47%
e

...
TODO : 홀짝 말고 ECG 데이터셋으로 직접 만들어보기